# Data Wrangling using Pandas

## What is Data Wrangling?

Data wrangling (also called data munging) is the process of transforming raw, messy data into a clean, structured format ready for analysis. In practice, data scientists spend **60-80% of their time** on this — it is the most important and most underestimated skill.

This notebook walks you through the entire wrangling pipeline: detecting and fixing missing values, removing duplicates, converting data types, standardizing text, reshaping data, combining multiple tables, and aggregating with GroupBy. Every technique is demonstrated on a realistic hospital patient dataset with intentionally messy data.

### The Healthcare Data Integration Disaster

A hospital merges patient data from three departments:

- **Emergency Department** entered data using **System A** (some fields optional)
- **Outpatient Clinic** used **System B** (different field names, different required fields)
- **Lab Results** came from **System C** (no patient name — only ID numbers)

**Result:** After merging, **35% of records had at least one missing field**.

**The danger:** A missing blood type in a patient record is not just a blank cell — it is a potential **transfusion error**. A duplicated patient ID means double billing — or worse, mixing up medication records.

| System | Department | Missing Fields | Issue |
|--------|-----------|---------------|-------|
| System A | Emergency | blood_group often blank | Optional field → missing data |
| System B | Outpatient | Different column names | Schema mismatch |
| System C | Lab Results | No patient name | ID-only → needs merge |

In [1]:
import pandas as pd

In [2]:
# ========================================
# SESSION DATASET: Hospital Patient Records
# This dataset has INTENTIONAL problems — your job is to find and fix them!
# Used throughout Sections 1-8
# ========================================

patients = pd.DataFrame({
    'patient_id': ['P001', 'P002', 'P003', 'P004', 'P005',
                   'P006', 'P007', 'P008', 'P002', 'P010'],
    'name': ['Rajesh Kumar', ' Priya Sharma ', 'ANITA DESAI',
             'vikram patel', 'Sneha Iyer', None,
             'Meera Joshi', 'karan singh', 'Priya Sharma', 'Divya Rao'],
    'age': [45, 32, None, 28, 56, 41, None, 35, 32, 29],
    'blood_group': ['A+', 'B+', 'O-', 'AB+', 'A-',
                    'B+', 'O+', None, 'B+', 'A+'],
    'admission_date': ['2024-01-15', '2024-01-16', '2024-01-16',
                       '15-01-2024', '2024-01-17', '2024-01-18',
                       '2024-01-19', '2024-01-20', '2024-01-20', '2024-01-21'],
    'bill_amount': [15000, 22500, None, 18500, 20000.50, 'N/A',
                    17999.99, 19500, 22500, 16000],
    'department': ['Cardiology', 'cardiology', 'Orthopedics',
                   'orthopedics', 'Cardiology', 'CARDIOLOGY',
                   'Orthopedics', 'cardiology', 'cardiology', 'Cardiology']
})

print(f"Dataset loaded: {patients.shape[0]} rows x {patients.shape[1]} columns")
print(f"Columns: {patients.columns.tolist()}")

Dataset loaded: 10 rows x 7 columns
Columns: ['patient_id', 'name', 'age', 'blood_group', 'admission_date', 'bill_amount', 'department']


In [3]:
patients

,patient_id,name,age,blood_group,admission_date,bill_amount,department
0,P001,Rajesh Kumar,45.0,A+,2024-01-15,15000,Cardiology
1,P002,Priya Sharma,32.0,B+,2024-01-16,22500,cardiology
2,P003,ANITA DESAI,NaN,O-,2024-01-16,None,Orthopedics
3,P004,vikram patel,28.0,AB+,15-01-2024,18500,orthopedics
4,P005,Sneha Iyer,56.0,A-,2024-01-17,20000.5,Cardiology
5,P006,None,41.0,B+,2024-01-18,N/A,CARDIOLOGY
6,P007,Meera Joshi,NaN,O+,2024-01-19,17999.99,Orthopedics
7,P008,karan singh,35.0,None,2024-01-20,19500,cardiology
8,P002,Priya Sharma,32.0,B+,2024-01-20,22500,cardiology
9,P010,Divya Rao,29.0,A+,2024-01-21,16000,Cardiology


<div style="background: #FEF9E7; color: black; border-left: 5px solid #F39C12; padding: 12px 15px; margin: 10px 0; border-radius: 4px;">
    <strong>Can You Spot All 7 Types of Problems?</strong><br><br>
    1. <strong>Missing values:</strong> <code>None</code> in name (row 5), age (rows 2, 6), blood_group (row 7), bill_amount (row 2)<br>
    2. <strong>String masquerading as missing:</strong> <code>'N/A'</code> in bill_amount (row 5) — looks missing but is actually a STRING<br>
    3. <strong>Duplicate record:</strong> Patient P002 (Priya Sharma) appears at rows 1 AND 8<br>
    4. <strong>Inconsistent name casing:</strong> 'Rajesh Kumar' vs 'ANITA DESAI' vs 'vikram patel' vs 'karan singh'<br>
    5. <strong>Leading/trailing whitespace:</strong> ' Priya Sharma ' has spaces on both sides<br>
    6. <strong>Inconsistent date formats:</strong> '2024-01-15' (ISO) vs '15-01-2024' (DD-MM-YYYY)<br>
    7. <strong>Inconsistent department casing:</strong> 'Cardiology' vs 'cardiology' vs 'CARDIOLOGY'
</div>

In [4]:
## Exploring the dataset.
print("==== Data Types ====")
print(patients.dtypes)

print("==== Dataset Info ====")
print(patients.info())

==== Data Types ====
patient_id         object
name               object
age               float64
blood_group        object
admission_date     object
bill_amount        object
department         object
dtype: object
==== Dataset Info ====
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   patient_id      10 non-null     object 
 1   name            9 non-null      object 
 2   age             8 non-null      float64
 3   blood_group     9 non-null      object 
 4   admission_date  10 non-null     object 
 5   bill_amount     9 non-null      object 
 6   department      10 non-null     object 
dtypes: float64(1), object(6)
memory usage: 692.0+ bytes
None



## Part 1: Handling Missing Values

### NaN vs None vs 'N/A' — Three Different Things

Before we start handling missing values, we need to understand that there are **three different representations** of "missing" that you will encounter:

- **`None`** — Python's built-in null object. Pandas converts this to `NaN` in numeric columns.
- **`np.nan`** — NumPy's "Not a Number" sentinel. This is the standard missing-value marker in Pandas.
- **`'N/A'`** (or `'null'`, `'None'`, `'-'`, `''`) — These are just regular **strings**. Pandas does NOT recognize them as missing unless you explicitly tell it to.

The first two are treated as missing by Pandas. The third is **not** — and that is where hidden bugs live.

In [5]:
# Three DIFFERENT representations of "missing" — only 2 are treated as NaN by Pandas
import numpy as np

val_none = None       # Python's None
val_nan = np.nan      # NumPy's Not a Number
val_na_str = 'N/A'    # Just a regular string!

print(f"None:  type = {type(val_none).__name__},  pd.isna() = {pd.isna(val_none)}")
print(f"NaN:   type = {type(val_nan).__name__}, pd.isna() = {pd.isna(val_nan)}")
print(f"'N/A': type = {type(val_na_str).__name__},   pd.isna() = {pd.isna(val_na_str)}")
print()
print("Key insight: Pandas treats None and NaN as missing, but 'N/A' is just text!")

None:  type = NoneType,  pd.isna() = True
NaN:   type = float, pd.isna() = True
'N/A': type = str,   pd.isna() = False

Key insight: Pandas treats None and NaN as missing, but 'N/A' is just text!


#### .isna() or .isnull() can be used to identify missing/null values

In [6]:
print(patients.isna())
print()
print(patients.isna().sum())
print()
print(patients.isnull().sum())
print()
print(f"Total Missing Values: {patients.isnull().sum().sum()}")

   patient_id   name    age  blood_group  admission_date  bill_amount  \
0       False  False  False        False           False        False   
1       False  False  False        False           False        False   
2       False  False   True        False           False         True   
3       False  False  False        False           False        False   
4       False  False  False        False           False        False   
5       False   True  False        False           False        False   
6       False  False   True        False           False        False   
7       False  False  False         True           False        False   
8       False  False  False        False           False        False   
9       False  False  False        False           False        False   

   department  
0       False  
1       False  
2       False  
3       False  
4       False  
5       False  
6       False  
7       False  
8       False  
9       False  

patient_id        0

In [7]:
percentage_missing = patients.isna().mean() * 100
for col, pct_missing in percentage_missing.items():
    status = 'OK' if pct_missing == 0 else f'{round(pct_missing, 2)}% Missing'
    print(f"{col:20s}: {status}")

patient_id          : OK
name                : 10.0% Missing
age                 : 20.0% Missing
blood_group         : 10.0% Missing
admission_date      : OK
bill_amount         : 10.0% Missing
department          : OK


### Three Strategies for Handling Missing Data

There is no single "correct" way to handle missing values. The right approach depends on **how much** data is missing, **why** it is missing, and **what** you plan to do with the data.

| Strategy | Method | When to Use |
|----------|--------|-------------|
| **Drop** | `.dropna()` | Small % missing, data is missing randomly, dataset is large enough |
| **Fill** | `.fillna()` | Need to preserve all rows, have a reasonable replacement value |
| **Flag** | Create indicator column | Missingness itself is informative (e.g., unanswered survey question) |

### Strategy 1: Drop rows with missing values — `.dropna()`

The simplest approach: remove any row that has a missing value. This is appropriate when:
- Only a small percentage of rows are affected
- The data is missing **randomly** (not systematically)
- Your dataset is large enough that losing some rows does not bias the results

In [8]:
# Strategy 1a: Drop ALL rows that have ANY missing value
patients_dropped_all = patients.dropna()
print(f"Before: {len(patients)} rows -> After: {len(patients_dropped_all)} rows")
print(f"Lost {len(patients) - len(patients_dropped_all)} rows ({(len(patients) - len(patients_dropped_all))/len(patients)*100:.0f}% of data)")

Before: 10 rows -> After: 6 rows
Lost 4 rows (40% of data)


In [9]:
# Strategy 1b: Drop rows only when SPECIFIC columns are missing
# More targeted -  we only care about missing age values
patients_dropped_age = patients.dropna(subset='age')
print(f"Before: {len(patients)} rows -> After {len(patients_dropped_age)} rows")
print(f"Only dropped rows where 'age' was missing")

Before: 10 rows -> After 8 rows
Only dropped rows where 'age' was missing


In [10]:
# We can also drop based on a threshold: keep rows with at least N non-null values
patients_thresh = patients.dropna(thresh=6)  # Keep rows with at least 6 non-null values
print(f"Before: {len(patients)} rows -> After: {len(patients_thresh)} rows")
print(f"Kept rows with at least 6 non-null values out of {len(patients.columns)} columns")

Before: 10 rows -> After: 9 rows
Kept rows with at least 6 non-null values out of 7 columns


### Strategy 2: Fill missing values — `.fillna()`

Instead of dropping rows, you can **replace** missing values with a substitute. Common fill strategies:

- **Constant value:** Fill with a known placeholder (e.g., `'Unknown'` for text, `0` for counts)
- **Statistical value:** Fill with the column's mean, median, or mode (for numeric data)
- **Forward/backward fill:** Use the previous or next row's value (for time series data)

In [11]:
# Strategy 2a: Fill with a constant value
patients_filled = patients.copy()
patients_filled['blood_group'] = patients_filled['blood_group'].fillna('Unknown')
print("Filled missing blood_group with 'Unknown':")
print(f"  Before: {patients['blood_group'].isna().sum()} missing")
print(f"  After:  {patients_filled['blood_group'].isna().sum()} missing")

Filled missing blood_group with 'Unknown':
  Before: 1 missing
  After:  0 missing


In [12]:
# Strategy 2b: Fill numeric columns with the mean (common for continuous data)
# First, we need to handle the 'N/A' string before we can compute a mean
# For now, let's work with the 'age' column which has proper NaN values
patients_filled_mean = patients.copy()
mean_age = patients_filled_mean['age'].mean()
patients_filled_mean['age'] = patients_filled_mean['age'].fillna(mean_age)
print(f"Mean age: {mean_age:.1f}")
print(f"Filled missing ages with mean value")
print(f"  Before: {patients['age'].isna().sum()} missing -> After: {patients_filled_mean['age'].isna().sum()} missing")

Mean age: 37.2
Filled missing ages with mean value
  Before: 2 missing -> After: 0 missing


In [13]:
# Strategy 2c: Forward fill (ffill) — use the PREVIOUS row's value
# Useful for time series data where values change slowly
patients_ffill = patients.copy()
patients_ffill['age'] = patients_ffill['age'].ffill()
print("Forward fill for 'age':")
print(f"  Row 2 (was None) -> now {patients_ffill.loc[2, 'age']} (copied from row 1)")
print(f"  Row 6 (was None) -> now {patients_ffill.loc[6, 'age']} (copied from row 5)")

Forward fill for 'age':
  Row 2 (was None) -> now 32.0 (copied from row 1)
  Row 6 (was None) -> now 41.0 (copied from row 5)


<div style="background: #FEF9E7; color: black; border-left: 5px solid #F39C12; padding: 12px 15px; margin: 10px 0; border-radius: 4px;">
    <strong>Warning — When NOT to Fill:</strong><br>
    Sometimes missing data IS the information. If a patient's blood_group is missing, filling it with 'A+' (the most common) could cause a <strong>fatal transfusion error</strong>. The right approach depends on the domain: fill numeric columns with mean/median for analysis, but leave critical medical fields as 'Unknown' or NaN.
</div>

### Strategy 3: The Hidden 'N/A' String Problem

This is one of the most common traps in real-world data. When someone enters `'N/A'` or `'null'` or `'-'` into a spreadsheet, it **looks** like a missing value to a human. But to Pandas, it is just a regular string — not missing at all.

Our `bill_amount` column has this exact problem: one value is `None` (actual NaN) and another is the string `'N/A'` (looks missing, but Pandas treats it as present text).

In [14]:
# The hidden problem: 'N/A' is NOT recognized as NaN
print(f"bill_amount dtype: {patients['bill_amount'].dtype}")  # object — should be numeric!
print(f"bill_amount values: {patients['bill_amount'].tolist()}")
print()
print("Notice: 'N/A' at index 5 is a STRING, not NaN")
print(f"pd.isna('N/A') = {pd.isna('N/A')}  <- Pandas does NOT treat 'N/A' string as missing!")

bill_amount dtype: object
bill_amount values: [15000, 22500, None, 18500, 20000.5, 'N/A', 17999.99, 19500, 22500, 16000]

Notice: 'N/A' at index 5 is a STRING, not NaN
pd.isna('N/A') = False  <- Pandas does NOT treat 'N/A' string as missing!


In [15]:
# Fix: Replace 'N/A' with actual NaN, then all missing values will be detectable.
patients_fixed = patients.copy()
patients_fixed['bill_amount'] = pd.to_numeric(patients_fixed['bill_amount'].replace('N/A', np.nan), errors = 'coerce')
print(f"Before fix: {patients['bill_amount'].isna().sum()} NaN detected")
print(f"After fix:  {patients_fixed['bill_amount'].isna().sum()} NaN detected")
print()
print("Now Pandas sees BOTH missing values in bill_amount!")
print(patients_fixed.dtypes)

Before fix: 1 NaN detected
After fix:  2 NaN detected

Now Pandas sees BOTH missing values in bill_amount!
patient_id         object
name               object
age               float64
blood_group        object
admission_date     object
bill_amount       float64
department         object
dtype: object


/tmp/ipykernel_45967/1640472311.py:3: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  patients_fixed['bill_amount'] = pd.to_numeric(patients_fixed['bill_amount'].replace('N/A', np.nan), errors = 'coerce')


<div style="background: #EAFAF1; color: black; border-left: 5px solid #2ECC71; padding: 12px 15px; margin: 10px 0; border-radius: 4px;">
    <strong>Best Practice:</strong> Always check for string representations of missing data. Common culprits include <code>'N/A'</code>, <code>'NA'</code>, <code>'n/a'</code>, <code>'null'</code>, <code>'None'</code>, <code>'-'</code>, <code>''</code> (empty string), and <code>'missing'</code>. A quick way to check: <code>df['column'].unique()</code> — scan the output for any suspicious strings.
</div>

## Part 2: Data Type Conversion

<div style="background: #F5F5F5; color: grey; border-left: 5px solid #999; padding: 12px 15px; margin: 10px 0; border-radius: 4px;">
    <strong>Analogy — Currency Conversion:</strong><br>
    Think of data types like currencies. The number '100' means different things as dollars, rupees, or yen — the raw digits are identical, but the interpretation changes everything. Similarly, <code>'15000'</code> as a string is just five characters you can concatenate and search. As a number, it is a value you can add, compare, and average. Same raw data, different type, completely different behavior.
</div>

In [16]:
patients.dtypes

,0
patient_id,object
name,object
age,float64
blood_group,object
admission_date,object
bill_amount,object
department,object


Notice that `bill_amount` is `object` (string) type — it should be numeric. And `admission_date` is also `object` — it should be datetime. We cannot do math on strings or date arithmetic on text!

In [17]:
# The problem: can't do math on string columns
try:
    avg_bill = patients['bill_amount'].mean()
    print(f"Average bill: {avg_bill}")
except TypeError as e:
    print(f"TypeError: {e}")
    print()
    print("bill_amount is stored as 'object' (string) — Pandas can't compute mean on strings!")
    print(f"Actual values: {patients['bill_amount'].tolist()}")
    print("The culprit: 'N/A' at index 5 forced the entire column to be stored as text")

TypeError: unsupported operand type(s) for +: 'float' and 'str'

bill_amount is stored as 'object' (string) — Pandas can't compute mean on strings!
Actual values: [15000, 22500, None, 18500, 20000.5, 'N/A', 17999.99, 19500, 22500, 16000]
The culprit: 'N/A' at index 5 forced the entire column to be stored as text


In [18]:
# The naive fix FAILS: .astype(float) can't handle 'N/A' or None
try:
    patients['bill_amount'].astype(float)
except (ValueError, TypeError) as e:
    print(f"Error: {e}")
    print()
    print("You can't directly convert a column containing 'N/A' strings to float!")

Error: could not convert string to float: 'N/A'

You can't directly convert a column containing 'N/A' strings to float!


<div style="background: #FEF9E7; color: black; border-left: 5px solid #F39C12; padding: 12px 15px; margin: 10px 0; border-radius: 4px;">
    <strong>Warning — Two-Step Pattern: Clean FIRST, Convert SECOND.</strong><br>
    If you try <code>.astype(float)</code> on a column containing 'N/A' or other non-numeric strings, you get a ValueError. The solution: (1) Replace problematic strings with NaN, (2) THEN convert the type.
</div>

In [19]:
# Step 1: Replace 'N/A' string with actual NaN
patients_clean = patients.copy()
patients_clean['bill_amount'] = patients_clean['bill_amount'].replace('N/A', np.nan)
print("Step 1 done: Replaced 'N/A' with NaN")
print(f"Values: {patients_clean['bill_amount'].tolist()}")

Step 1 done: Replaced 'N/A' with NaN
Values: [15000.0, 22500.0, nan, 18500.0, 20000.5, nan, 17999.99, 19500.0, 22500.0, 16000.0]


/tmp/ipykernel_45967/1832925602.py:3: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  patients_clean['bill_amount'] = patients_clean['bill_amount'].replace('N/A', np.nan)


In [20]:
# Step 2: Safe conversion with pd.to_numeric(errors='coerce')
# 'coerce' means: if a value can't be converted, make it NaN instead of crashing
patients_clean['bill_amount'] = pd.to_numeric(patients_clean['bill_amount'], errors='coerce')
print(f"bill_amount dtype: {patients_clean['bill_amount'].dtype}")
print(f"Mean bill: Rs. {patients_clean['bill_amount'].mean():,.2f}")
print(f"Max bill:  Rs. {patients_clean['bill_amount'].max():,.2f}")
print()
print("Now we can do math!")

bill_amount dtype: float64
Mean bill: Rs. 19,000.06
Max bill:  Rs. 22,500.00

Now we can do math!


### Date Conversion — `pd.to_datetime()`

Our `admission_date` column has a tricky problem: **two different date formats** in the same column.

- Most rows use ISO format: `'2024-01-15'` (YYYY-MM-DD)
- One row uses a different format: `'15-01-2024'` (DD-MM-YYYY)

This is extremely common when data comes from multiple sources or is manually entered.

In [21]:
# The date problem: two different formats in the same column
print("Current admission_date values:")
for i, date in enumerate(patients['admission_date']):
    fmt = "DD-MM-YYYY" if date.startswith(('0','1','2','3')) and date[2] == '-' else "YYYY-MM-DD"
    print(f"  Row {i}: '{date}'  <- {fmt}")

Current admission_date values:
  Row 0: '2024-01-15'  <- YYYY-MM-DD
  Row 1: '2024-01-16'  <- YYYY-MM-DD
  Row 2: '2024-01-16'  <- YYYY-MM-DD
  Row 3: '15-01-2024'  <- DD-MM-YYYY
  Row 4: '2024-01-17'  <- YYYY-MM-DD
  Row 5: '2024-01-18'  <- YYYY-MM-DD
  Row 6: '2024-01-19'  <- YYYY-MM-DD
  Row 7: '2024-01-20'  <- YYYY-MM-DD
  Row 8: '2024-01-20'  <- YYYY-MM-DD
  Row 9: '2024-01-21'  <- YYYY-MM-DD


In [22]:
# pd.to_datetime() with format='mixed' handles multiple formats
patients_clean['admission_date'] = pd.to_datetime(
    patients['admission_date'],
    dayfirst=True, # Tell Pandas that day comes first in ambiguous dates
    format='mixed' # Allow mixed formats in the same column
)
print(f"admission_date dtype: {patients_clean['admission_date'].dtype}")
print()
print("Converted dates:")
for i, date in enumerate(patients_clean['admission_date']):
    print(f"  Row {i}: {date.strftime('%Y-%m-%d')}")

admission_date dtype: datetime64[ns]

Converted dates:
  Row 0: 2024-01-15
  Row 1: 2024-01-16
  Row 2: 2024-01-16
  Row 3: 2024-01-15
  Row 4: 2024-01-17
  Row 5: 2024-01-18
  Row 6: 2024-01-19
  Row 7: 2024-01-20
  Row 8: 2024-01-20
  Row 9: 2024-01-21


In [23]:
# Now we can do date operations!
print(f"Date range: {patients_clean['admission_date'].min().strftime('%Y-%m-%d')} to {patients_clean['admission_date'].max().strftime('%Y-%m-%d')}")
print(f"Span: {(patients_clean['admission_date'].max() - patients_clean['admission_date'].min()).days} days")

Date range: 2024-01-15 to 2024-01-21
Span: 6 days


<div style="background: #EBF5FB; color: grey; border-left: 5px solid #3498DB; padding: 12px 15px; margin: 10px 0; border-radius: 4px;">
    <strong>Key Concept — The dtype Hierarchy:</strong><br><br>
    <code>object</code> → <code>numeric</code> / <code>datetime</code> → <code>categorical</code><br><br>
    When you load a CSV, Pandas guesses data types. If ANY value in a column is non-numeric (like 'N/A'), the ENTIRE column becomes <code>object</code> (string). This is why <code>.dtypes</code> is always your first diagnostic check — if a numeric column shows as <code>object</code>, something is wrong.
</div>

## Part 3: String Operations - The .str Accessor

<div style="background: #F5F5F5; color: grey; border-left: 5px solid #999; padding: 12px 15px; margin: 10px 0; border-radius: 4px;">
    <strong>Analogy — Find-and-Replace for Entire Columns:</strong><br>
    Think of the <code>.str</code> accessor as <strong>Find-and-Replace in a word processor</strong> — but for an entire column at once. Instead of fixing text one cell at a time (the Excel way), Pandas lets you apply a transformation to every value in a column with one line of code.
</div>

In [24]:
# The problem: inconsistent casing makes analysis WRONG
# Pandas treats 'Cardiology', 'cardiology', and 'CARDIOLOGY' as THREE different departments!
patients['department'].value_counts()

,count
department,
Cardiology,3
cardiology,3
Orthopedics,2
orthopedics,1
CARDIOLOGY,1


### The `.str` Accessor

Any string method you know from Python (`lower()`, `upper()`, `strip()`, `replace()`, `contains()`) can be applied to an **entire column** by putting `.str.` before it.

| Python String Method | Pandas Column Equivalent | What It Does |
|---------------------|-------------------------|-------------|
| `"hello".lower()` | `df['col'].str.lower()` | Convert to lowercase |
| `"hello".upper()` | `df['col'].str.upper()` | Convert to UPPERCASE |
| `"hello".title()` | `df['col'].str.title()` | Convert to Title Case |
| `"hello".strip()` | `df['col'].str.strip()` | Remove leading/trailing whitespace |
| `"lo" in "hello"` | `df['col'].str.contains("lo")` | Check if substring exists |

In [25]:
# .str.lower() — convert all values to lowercase
patients['department'].str.lower()

,department
0,cardiology
1,cardiology
2,orthopedics
3,orthopedics
4,cardiology
5,cardiology
6,orthopedics
7,cardiology
8,cardiology
9,cardiology


In [26]:
# .str.upper() — convert all values to UPPERCASE
patients['department'].str.upper()

,department
0,CARDIOLOGY
1,CARDIOLOGY
2,ORTHOPEDICS
3,ORTHOPEDICS
4,CARDIOLOGY
5,CARDIOLOGY
6,ORTHOPEDICS
7,CARDIOLOGY
8,CARDIOLOGY
9,CARDIOLOGY


In [27]:
# .str.title() — convert to Title Case (capitalize first letter of each word)
# This is the preferred format for proper nouns and department names
patients['department'].str.title()

,department
0,Cardiology
1,Cardiology
2,Orthopedics
3,Orthopedics
4,Cardiology
5,Cardiology
6,Orthopedics
7,Cardiology
8,Cardiology
9,Cardiology


In [28]:
# .str.strip() — remove leading and trailing whitespace
# Look at ' Priya Sharma ' — it has spaces on BOTH sides
print(f"Before strip: '{patients.loc[1, 'name']}'  (length: {len(patients.loc[1, 'name'])})")
print(f"After strip:  '{patients.loc[1, 'name'].strip()}'  (length: {len(patients.loc[1, 'name'].strip())})")

Before strip: ' Priya Sharma '  (length: 14)
After strip:  'Priya Sharma'  (length: 12)


<div style="background: #FEF9E7; color: black; border-left: 5px solid #F39C12; padding: 12px 15px; margin: 10px 0; border-radius: 4px;">
    <strong>Warning — Always strip BEFORE casing!</strong><br>
    <code>' Priya Sharma '.title()</code> produces <code>' Priya Sharma '</code> — the leading space REMAINS and causes merge/GroupBy problems. The correct order: <code>.str.strip().str.title()</code> — strip whitespace first, THEN standardize case.
</div>

In [29]:
# The correct pattern: strip → then case standardize
# Apply to the 'name' and 'department' columns
patients_str_clean = patients.copy()
patients_str_clean['name'] = patients_str_clean['name'].str.strip().str.title()
patients_str_clean['department'] = patients_str_clean['department'].str.strip().str.title()

print("Before cleaning:")
print(f"  Unique departments: {patients['department'].nunique()} → {patients['department'].unique().tolist()}")
print()
print("After cleaning:")
print(f"  Unique departments: {patients_str_clean['department'].nunique()} → {patients_str_clean['department'].unique().tolist()}")

Before cleaning:
  Unique departments: 5 → ['Cardiology', 'cardiology', 'Orthopedics', 'orthopedics', 'CARDIOLOGY']

After cleaning:
  Unique departments: 2 → ['Cardiology', 'Orthopedics']


In [30]:
# .str.contains() — search for a substring within each value
# Find all patients in Cardiology-related departments (case-insensitive)
cardio_mask = patients['department'].str.contains('cardio', case=False, na=False)
print(f"Cardiology patients: {cardio_mask.sum()} out of {len(patients)}")

Cardiology patients: 7 out of 10


In [31]:
# Display the Cardiology patients
patients[cardio_mask][['patient_id', 'name', 'department']]

,patient_id,name,department
0,P001,Rajesh Kumar,Cardiology
1,P002,Priya Sharma,cardiology
4,P005,Sneha Iyer,Cardiology
5,P006,None,CARDIOLOGY
7,P008,karan singh,cardiology
8,P002,Priya Sharma,cardiology
9,P010,Divya Rao,Cardiology


<div style="background: #EAFAF1; color: black; border-left: 5px solid #2ECC71; padding: 12px 15px; margin: 10px 0; border-radius: 4px;">
    <strong>Best Practice — The Complete String Cleaning Chain:</strong><br>
    <code>.str.strip()</code> (remove whitespace) → <code>.str.title()</code> or <code>.str.lower()</code> (standardize case) → <code>.str.replace()</code> (fix specific values). Always apply in this order for consistent results.
</div>

## Part 4: Handling Duplicates

<div style="background: #F5F5F5; color: grey; border-left: 5px solid #999; padding: 12px 15px; margin: 10px 0; border-radius: 4px;">
    <strong>Analogy — Hospital Wristband:</strong><br>
    Imagine a patient arrives at the emergency department. Due to a system glitch, two wristbands are printed with the same patient ID. Now: The nurse scans wristband 1 and records vitals. A different nurse scans wristband 2 and records vitals AGAIN. The billing system counts TWO admissions — the patient is billed twice. Duplicate records don't just waste storage — they corrupt every aggregation, average, and count you compute.
</div>

In [32]:
# Detect duplicates: .duplicated() returns True for duplicate records
patients.duplicated()

,0
0,False
1,False
2,False
3,False
4,False
5,False
6,False
7,False
8,False
9,False


In [33]:
# How many duplicates?
print(f"Number of duplicate rows: {patients.duplicated().sum()}")

Number of duplicate rows: 0


In [34]:
# Show the duplicate rows — which ones are copies?
patients[patients.duplicated(keep=False)]  # keep=False marks ALL copies, not just the second one

,patient_id,name,age,blood_group,admission_date,bill_amount,department


### Understanding the `keep` Parameter

The `keep` parameter controls which occurrence is marked as a duplicate:

- **`keep='first'`** (default): Marks the **second** (and subsequent) occurrence as duplicate. The first occurrence is kept.
- **`keep='last'`**: Marks the **first** occurrence as duplicate. The last occurrence is kept.
- **`keep=False`**: Marks **ALL** occurrences as duplicates — useful for inspection to see every copy.

In [35]:
# Compare keep options
print("keep='first' (default):", patients.duplicated(keep='first').sum(), "duplicates")
print("keep='last':           ", patients.duplicated(keep='last').sum(), "duplicates")
print("keep=False:            ", patients.duplicated(keep=False).sum(), "duplicates (marks ALL copies)")

keep='first' (default): 0 duplicates
keep='last':            0 duplicates
keep=False:             0 duplicates (marks ALL copies)


In [36]:
# Check duplicates on a SPECIFIC column (more useful in practice)
# Two patients with the same patient_id should NOT exist
patients.duplicated(subset=['patient_id'])

,0
0,False
1,False
2,False
3,False
4,False
5,False
6,False
7,False
8,True
9,False


In [37]:
# Show which patient_ids are duplicated
print("Duplicated patient IDs:")
dup_ids = patients[patients.duplicated(subset=['patient_id'], keep=False)]
print(f"Found {len(dup_ids)} rows with duplicate patient_id")

Duplicated patient IDs:
Found 2 rows with duplicate patient_id


In [38]:
dup_ids[['patient_id', 'name', 'age', 'department']]

,patient_id,name,age,department
1,P002,Priya Sharma,32.0,cardiology
8,P002,Priya Sharma,32.0,cardiology


In [39]:
# Remove duplicates — keep the FIRST occurrence
patients_deduped = patients.drop_duplicates(subset=['patient_id'], keep='first')
print(f"Before: {len(patients)} rows -> After: {len(patients_deduped)} rows")
print(f"Removed {len(patients) - len(patients_deduped)} duplicate(s)")

Before: 10 rows -> After: 9 rows
Removed 1 duplicate(s)


<div style="background: #EBF5FB; color: grey; border-left: 5px solid #3498DB; padding: 12px 15px; margin: 10px 0; border-radius: 4px;">
    <strong>Key Concept — Exact vs Partial Duplicates:</strong><br>
    <strong>Exact duplicates</strong> are rows where EVERY column matches (use <code>.duplicated()</code> with no <code>subset</code>). <strong>Partial duplicates</strong> share the same key column (e.g., patient_id) but may differ in other fields — these are trickier and require domain knowledge to resolve. In our example, P002 is an exact duplicate — both rows are identical.
</div>

<div style="background: #EAFAF1; color: black; border-left: 5px solid #2ECC71; padding: 12px 15px; margin: 10px 0; border-radius: 4px;">
    <strong>Best Practice:</strong> Always check for duplicates BEFORE and AFTER merging. Merging two tables can CREATE duplicates if the join key has a many-to-many relationship. A quick sanity check: compare <code>len(df)</code> before and after the merge.
</div>

## Part 5: Tidy Data Principles

<div style="background: #F5F5F5; color: grey; border-left: 5px solid #999; padding: 12px 15px; margin: 10px 0; border-radius: 4px;">
    <strong>Analogy — Filing Cabinet vs Loose Papers:</strong><br>
    Think of tidy data like a well-organized <strong>filing cabinet</strong> vs a drawer full of loose papers. Filing cabinet: each drawer is labeled (column), each folder inside is one person's file (row), and you have separate cabinets for HR, Finance, Sales (separate tables). Loose-paper drawer: everything is mixed together — you have to dig through the whole pile every time you need something.
</div>

<div style="background: #EBF5FB; color: grey; border-left: 5px solid #3498DB; padding: 12px 15px; margin: 10px 0; border-radius: 4px;">
    <strong>The Three Rules of Tidy Data (Hadley Wickham, 2014):</strong><br><br>
    1. <strong>Each variable</strong> forms its own <strong>column</strong><br>
    2. <strong>Each observation</strong> forms its own <strong>row</strong><br>
    3. <strong>Each value</strong> occupies its own <strong>cell</strong><br><br>
    If your data follows these rules, every Pandas operation (filter, group, merge, plot) becomes straightforward. If it doesn't, you need to <strong>reshape</strong> it first.
</div>

In [40]:
# Example of UNTIDY (wide) data — months are stored as columns
sales_wide = pd.DataFrame({
    'salesperson': ['Priya', 'Rahul', 'Anita'],
    'Jan': [45000, 38000, 52000],
    'Feb': [48000, 41000, 49000],
    'Mar': [51000, 43000, 55000]
})

print("UNTIDY (wide format) — months are column names, not values:")

UNTIDY (wide format) — months are column names, not values:


In [41]:
sales_wide

,salesperson,Jan,Feb,Mar
0,Priya,45000,48000,51000
1,Rahul,38000,41000,43000
2,Anita,52000,49000,55000


In [42]:
# pd.melt() — convert wide format to tidy (long) format
# id_vars: columns to keep as-is
# var_name: name for the column created from old column headers
# value_name: name for the column created from old values
sales_tidy = pd.melt(
    sales_wide,
    id_vars=['salesperson'],
    var_name='month',
    value_name='sales'
)

print(f"Wide: {sales_wide.shape} → Tidy: {sales_tidy.shape}")
print("Now each row is ONE observation: one salesperson's sales in one month")

Wide: (3, 4) → Tidy: (9, 3)
Now each row is ONE observation: one salesperson's sales in one month


In [43]:
sales_tidy

,salesperson,month,sales
0,Priya,Jan,45000
1,Rahul,Jan,38000
2,Anita,Jan,52000
3,Priya,Feb,48000
4,Rahul,Feb,41000
5,Anita,Feb,49000
6,Priya,Mar,51000
7,Rahul,Mar,43000
8,Anita,Mar,55000


In [44]:
# pivot_table() — convert tidy (long) back to wide format
# Useful for presentation/reporting
sales_back_to_wide = sales_tidy.pivot_table(
    values='sales',
    index='salesperson',
    columns='month'
)

print("Back to wide format (for presentation):")

Back to wide format (for presentation):


In [45]:
sales_back_to_wide

month,Feb,Jan,Mar
salesperson,,,
Anita,49000.0,52000.0,55000.0
Priya,48000.0,45000.0,51000.0
Rahul,41000.0,38000.0,43000.0


<div style="background: #EAFAF1; color: black; border-left: 5px solid #2ECC71; padding: 12px 15px; margin: 10px 0; border-radius: 4px;">
    <strong>Best Practice:</strong> <strong>Tidy for analysis, wide for presentation.</strong> Use <code>pd.melt()</code> to reshape wide data into tidy format before analysis (filtering, grouping, plotting). Use <code>pd.pivot_table()</code> to reshape tidy data back to wide for final reports and dashboards.
</div>

## Part 6: Combining DataFrames - pd.concat()

<div style="background: #F5F5F5; color: grey; border-left: 5px solid #999; padding: 12px 15px; margin: 10px 0; border-radius: 4px;">
    <strong>Analogy — Cafeteria Trays:</strong><br>
    Think of <code>pd.concat()</code> like stacking <strong>cafeteria trays</strong>. <strong>Vertical stacking (axis=0):</strong> Putting one tray on top of another — more food items (rows), same categories (columns). <strong>Horizontal stacking (axis=1):</strong> Putting trays side by side — same items (rows), more information about each (columns).
</div>